In [ ]:
!pip install groq yfinance requests beautifulsoup4 pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
import groq, yfinance as yf, requests, json, time, pandas as pd, numpy as np
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from IPython.display import display, HTML

# ── Config ──────────────────────────────────────────────────────
GROQ_KEY  = "gsk_your_key_here"   # 👈 paste your Groq key
client    = groq.Groq(api_key=GROQ_KEY)
MODEL     = "meta-llama/llama-4-scout-17b-16e-instruct"
TODAY         = datetime.now().strftime("%Y-%m-%d")
DAY_AFTER_TMR = (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d")

# ── Data Fetchers ────────────────────────────────────────────────
def fetch_news(ticker, company):
    articles, headers = [], {"User-Agent": "Mozilla/5.0"}
    try:
        url  = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US"
        soup = BeautifulSoup(requests.get(url, headers=headers, timeout=10).content, "xml")
        for item in soup.find_all("item")[:8]:
            articles.append({"source":"Yahoo Finance",
                "title":   item.find("title").text if item.find("title") else "",
                "summary": item.find("description").text if item.find("description") else "",
                "date":    TODAY})
    except: pass
    try:
        url  = f"https://finviz.com/quote.ashx?t={ticker}"
        soup = BeautifulSoup(requests.get(url, headers=headers, timeout=10).text, "html.parser")
        table = soup.find("table", class_="fullview-news-outer")
        if table:
            for row in table.find_all("tr")[:8]:
                a = row.find("a")
                if a: articles.append({"source":"Finviz","title":a.text.strip(),"summary":a.text.strip(),"date":TODAY})
    except: pass
    print(f"   📰 {len(articles)} articles fetched")
    return articles

def fetch_price_data(ticker):
    stock = yf.Ticker(ticker)
    hist  = stock.history(period="6mo")
    info  = stock.info
    if hist.empty: return {}
    closes = hist["Close"]
    sma20  = closes.rolling(20).mean().iloc[-1]
    sma50  = closes.rolling(50).mean().iloc[-1]
    delta  = closes.diff()
    rs     = delta.clip(lower=0).rolling(14).mean() / (-delta.clip(upper=0)).rolling(14).mean()
    rsi    = (100 - (100/(1+rs))).iloc[-1]
    ema12  = closes.ewm(span=12).mean()
    ema26  = closes.ewm(span=26).mean()
    macd   = (ema12-ema26).iloc[-1]
    signal = (ema12-ema26).ewm(span=9).mean().iloc[-1]
    bb_mid = closes.rolling(20).mean().iloc[-1]
    bb_std = closes.rolling(20).std().iloc[-1]
    price_now = closes.iloc[-1]
    return {
        "ticker": ticker,
        "company": info.get("longName", ticker),
        "sector":  info.get("sector","N/A"),
        "current_price": round(price_now,2),
        "pe_ratio": info.get("trailingPE","N/A"),
        "beta":     info.get("beta","N/A"),
        "52w_high": info.get("fiftyTwoWeekHigh","N/A"),
        "52w_low":  info.get("fiftyTwoWeekLow","N/A"),
        "technicals": {
            "sma20": round(sma20,2), "sma50": round(sma50,2),
            "rsi14": round(rsi,2),
            "macd":  round(macd,4),  "macd_signal": round(signal,4),
            "bb_upper": round(bb_mid+2*bb_std,2),
            "bb_lower": round(bb_mid-2*bb_std,2),
            "volume_ratio": round(hist["Volume"].iloc[-1]/hist["Volume"].rolling(20).mean().iloc[-1],2),
        },
        "price_changes": {
            "1_week_%":  round((price_now - closes.iloc[-5])/closes.iloc[-5]*100,2) if len(closes)>=5 else 0,
            "1_month_%": round((price_now - closes.iloc[-21])/closes.iloc[-21]*100,2) if len(closes)>=21 else 0,
        },
    }

def ask_llm(system, user):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}],
        max_tokens=1024, temperature=0.1)
    raw = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
    try:    return json.loads(raw)
    except: return {"raw": raw}

# ── Agent 1: News Sentiment ──────────────────────────────────────
def agent1(ticker, company, articles):
    print("\n📰 Agent 1 — News Sentiment Analyst...")
    news_text = "\n".join([f"[{a['source']}] {a['title']}" for a in articles]) or "No news found."
    result = ask_llm(
        "You are a Wall Street news sentiment analyst. Reply with valid JSON only.",
        f"""Analyze news for {ticker} ({company}) and predict bias for {DAY_AFTER_TMR}.
News:\n{news_text}

Reply ONLY with JSON:
{{"overall_sentiment":"BULLISH|BEARISH|NEUTRAL","sentiment_score":<-1.0 to 1.0>,
"confidence":<0.0-1.0>,"key_positives":["..."],"key_negatives":["..."],
"catalyst_present":true/false,"catalyst":"...","reasoning":"...","bias":"{DAY_AFTER_TMR} UP|DOWN|SIDEWAYS"}}"""
    )
    print(f"   ✅ Sentiment: {result.get('overall_sentiment')} | Score: {result.get('sentiment_score')}")
    return result

# ── Agent 2: Technical Predictor ────────────────────────────────
def agent2(ticker, price_data):
    print("\n📊 Agent 2 — Technical Price Predictor...")
    t = price_data["technicals"]
    result = ask_llm(
        "You are a quantitative analyst. Reply with valid JSON only.",
        f"""Predict {ticker} movement for {DAY_AFTER_TMR} using technical data.

Price: ${price_data['current_price']} | RSI: {t['rsi14']} | MACD: {t['macd']} vs Signal: {t['macd_signal']}
SMA20: {t['sma20']} | SMA50: {t['sma50']} | BB Upper: {t['bb_upper']} | BB Lower: {t['bb_lower']}
Volume Ratio: {t['volume_ratio']}x | 1W Change: {price_data['price_changes']['1_week_%']}% | 1M Change: {price_data['price_changes']['1_month_%']}%
Beta: {price_data['beta']} | P/E: {price_data['pe_ratio']} | 52W High: {price_data['52w_high']} | 52W Low: {price_data['52w_low']}

Reply ONLY with JSON:
{{"technical_bias":"BULLISH|BEARISH|NEUTRAL","technical_score":<-1.0 to 1.0>,
"confidence":<0.0-1.0>,"trend":"UPTREND|DOWNTREND|SIDEWAYS",
"predicted_price_low":<float>,"predicted_price_mid":<float>,"predicted_price_high":<float>,
"predicted_change_%":<float>,"support":<float>,"resistance":<float>,
"rsi_signal":"OVERBOUGHT|OVERSOLD|NEUTRAL","risk":"HIGH|MEDIUM|LOW",
"stop_loss":<float>,"reasoning":"...","direction":"{DAY_AFTER_TMR} UP|DOWN|SIDEWAYS"}}"""
    )
    print(f"   ✅ Bias: {result.get('technical_bias')} | Predicted: ${result.get('predicted_price_mid')}")
    return result

# ── Agent 3: Master Synthesizer ──────────────────────────────────
def agent3(ticker, company, sentiment, technical, price_data):
    print("\n🧠 Agent 3 — Master Synthesizer...")
    s = sentiment.get("sentiment_score",0) * sentiment.get("confidence",0.5)
    t = technical.get("technical_score",0) * technical.get("confidence",0.5)
    combined = round(s*0.4 + t*0.6, 4)
    result = ask_llm(
        "You are Chief Investment Strategist. Reply with valid JSON only.",
        f"""Synthesize these two reports for {ticker} ({company}) and give FINAL verdict for {DAY_AFTER_TMR}.

AGENT 1 (News): {json.dumps(sentiment)}
AGENT 2 (Technical): {json.dumps(technical)}
COMBINED SCORE (40% news + 60% technical): {combined}
CURRENT PRICE: ${price_data['current_price']}

Reply ONLY with JSON:
{{"final_verdict":"STRONG BUY|BUY|HOLD|SELL|STRONG SELL","final_direction":"UP|DOWN|SIDEWAYS",
"combined_score":{combined},"overall_confidence":<0.0-1.0>,
"predicted_price":<float>,"predicted_change_%":<float>,
"predicted_range_low":<float>,"predicted_range_high":<float>,
"agents_agree":true/false,"conflict":"...","primary_risk":"...","opportunity":"...",
"final_reasoning":"3-4 sentences","disclaimer":"AI analysis for educational purposes only. Not financial advice."}}"""
    )
    print(f"   ✅ Verdict: {result.get('final_verdict')} | Direction: {result.get('final_direction')}")
    return result

# ── Display Report ───────────────────────────────────────────────
def display_report(sentiment, technical, verdict, price_data):
    v     = verdict.get("final_verdict","HOLD")
    d     = verdict.get("final_direction","SIDEWAYS")
    color = {"STRONG BUY":"#00C853","BUY":"#69F0AE","HOLD":"#FFD740","SELL":"#FF6D00","STRONG SELL":"#D50000"}.get(v,"#FFD740")
    emoji = {"UP":"📈","DOWN":"📉","SIDEWAYS":"➡️"}.get(d,"➡️")
    conf  = round(verdict.get("overall_confidence",0.5)*100,1)
    display(HTML(f"""
    <div style="font-family:Arial;max-width:780px;margin:auto;">
      <div style="background:#1a1a2e;color:white;padding:20px;border-radius:12px 12px 0 0;text-align:center;">
        <h2 style="margin:0;">🏦 MULTI-AGENT STOCK ANALYSIS</h2>
        <p style="color:#aaa;margin:4px">{TODAY} → Prediction for {DAY_AFTER_TMR}</p>
        <h3 style="color:#00d4ff;margin:6px 0;">{price_data.get('ticker')} — {price_data.get('company')}</h3>
      </div>
      <div style="background:{color};padding:20px;text-align:center;">
        <h1 style="margin:0;font-size:34px;">{emoji} {v}</h1>
        <p style="font-size:18px;margin:6px 0;">Current: <b>${price_data.get('current_price')}</b> → Predicted: <b>${verdict.get('predicted_price')}</b> (<b>{verdict.get('predicted_change_%')}%</b>)</p>
        <p style="margin:4px 0;">Range: ${verdict.get('predicted_range_low')} – ${verdict.get('predicted_range_high')} | Confidence: <b>{conf}%</b></p>
      </div>
      <div style="display:flex;gap:10px;padding:14px;background:#f5f5f5;">
        <div style="flex:1;background:white;border-radius:10px;padding:14px;border-top:4px solid #2196F3;">
          <h4 style="color:#2196F3;margin:0 0 8px;">📰 Agent 1 — News</h4>
          <p><b>Sentiment:</b> {sentiment.get('overall_sentiment')}</p>
          <p><b>Score:</b> {sentiment.get('sentiment_score')} | <b>Conf:</b> {round(sentiment.get('confidence',0)*100)}%</p>
          <p><b>Catalyst:</b> {'✅' if sentiment.get('catalyst_present') else '❌'} {sentiment.get('catalyst','')}</p>
          <p style="font-size:12px;color:#555;">{sentiment.get('reasoning','')}</p>
        </div>
        <div style="flex:1;background:white;border-radius:10px;padding:14px;border-top:4px solid #9C27B0;">
          <h4 style="color:#9C27B0;margin:0 0 8px;">📊 Agent 2 — Technical</h4>
          <p><b>Bias:</b> {technical.get('technical_bias')} | <b>Trend:</b> {technical.get('trend')}</p>
          <p><b>RSI:</b> {technical.get('rsi_signal')} | <b>Risk:</b> {technical.get('risk')}</p>
          <p><b>Support:</b> ${technical.get('support')} | <b>Resistance:</b> ${technical.get('resistance')}</p>
          <p style="font-size:12px;color:#555;">{technical.get('reasoning','')}</p>
        </div>
      </div>
      <div style="background:white;padding:16px;margin:0 14px 14px;">
        <h4>🧠 Agent 3 — Final Synthesis</h4>
        <p><b>Score:</b> {verdict.get('combined_score')} (40% News + 60% Technical) | <b>Agents Agree:</b> {'✅' if verdict.get('agents_agree') else '⚠️ Conflict'}</p>
        <p><b>Risk:</b> {verdict.get('primary_risk')}</p>
        <p><b>Opportunity:</b> {verdict.get('opportunity')}</p>
        <p style="background:#f9f9f9;padding:10px;border-radius:6px;border-left:3px solid #333;">{verdict.get('final_reasoning')}</p>
      </div>
      <div style="background:#ffebee;padding:10px;border-radius:0 0 12px 12px;font-size:11px;color:#c62828;text-align:center;">
        ⚠️ {verdict.get('disclaimer')}
      </div>
    </div>"""))

# ── Main Orchestrator ────────────────────────────────────────────
def analyze_stock(ticker):
    ticker = ticker.upper().strip()
    print(f"\n{'═'*55}\n  🏦 ANALYZING: {ticker} | Target: {DAY_AFTER_TMR}\n{'═'*55}")
    print("\n⬇️  Fetching price data...")
    price_data = fetch_price_data(ticker)
    if not price_data:
        print(f"❌ Could not fetch data for {ticker}")
        return
    print(f"   ✅ {price_data['company']} | ${price_data['current_price']}")
    print("\n⬇️  Fetching news...")
    articles  = fetch_news(ticker, price_data['company'])
    s = agent1(ticker, price_data['company'], articles); time.sleep(1)
    t = agent2(ticker, price_data);                      time.sleep(1)
    v = agent3(ticker, price_data['company'], s, t, price_data)
    print(f"\n{'═'*55}\n  📋 FINAL REPORT\n{'═'*55}\n")
    display_report(s, t, v, price_data)
    return v

print("✅ Loaded! Run: analyze_stock('AAPL')")

✅ Loaded! Run: analyze_stock('AAPL')


In [ ]:

import groq, yfinance as yf, requests, json, time, pandas as pd, numpy as np
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from IPython.display import display, HTML

TODAY         = datetime.now().strftime("%Y-%m-%d")
DAY_AFTER_TMR = (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d")
HEADERS       = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

GROQ_KEY = "YOUR_GROQ_KEY_HERE"   # 👈 paste your key
client   = groq.Groq(api_key=GROQ_KEY)
MODEL    = "meta-llama/llama-4-scout-17b-16e-instruct"


# ── Fetch Insider Transactions via yFinance ───────────────────────
def fetch_insider_transactions(ticker: str) -> list:
    """Pull recent insider buy/sell from yFinance (free, no API key)."""
    try:
        stock = yf.Ticker(ticker)
        insiders = stock.insider_transactions
        if insiders is None or insiders.empty:
            return []

        results = []
        for _, row in insiders.head(20).iterrows():
            results.append({
                "date":       str(row.get("Start Date", ""))[:10],
                "insider":    str(row.get("Insider", "")),
                "title":      str(row.get("Position", "")),
                "transaction":str(row.get("Transaction", "")),
                "shares":     int(row.get("Shares", 0)) if pd.notna(row.get("Shares")) else 0,
                "value_usd":  float(row.get("Value", 0)) if pd.notna(row.get("Value")) else 0,
                "shares_total": int(row.get("Shares Total", 0)) if pd.notna(row.get("Shares Total")) else 0,
            })
        print(f"   ✅ Insider Transactions: {len(results)} records")
        return results
    except Exception as e:
        print(f"   ⚠️  Insider transactions: {e}")
        return []


# ── Fetch Institutional Holders via yFinance ──────────────────────
def fetch_institutional_holders(ticker: str) -> list:
    """Pull top institutional/hedge fund holders from yFinance."""
    try:
        stock = yf.Ticker(ticker)
        inst  = stock.institutional_holders
        if inst is None or inst.empty:
            return []

        results = []
        for _, row in inst.head(15).iterrows():
            results.append({
                "holder":       str(row.get("Holder", "")),
                "shares":       int(row.get("Shares", 0)) if pd.notna(row.get("Shares")) else 0,
                "date_reported":str(row.get("Date Reported", ""))[:10],
                "pct_out":      float(row.get("% Out", 0)) if pd.notna(row.get("% Out")) else 0,
                "value_usd":    float(row.get("Value", 0)) if pd.notna(row.get("Value")) else 0,
            })
        print(f"   ✅ Institutional Holders: {len(results)} funds")
        return results
    except Exception as e:
        print(f"   ⚠️  Institutional holders: {e}")
        return []


# ── Fetch Mutual Fund Holders ─────────────────────────────────────
def fetch_mutualfund_holders(ticker: str) -> list:
    """Pull top mutual fund holders from yFinance."""
    try:
        stock = yf.Ticker(ticker)
        mf    = stock.mutualfund_holders
        if mf is None or mf.empty:
            return []

        results = []
        for _, row in mf.head(10).iterrows():
            results.append({
                "holder":       str(row.get("Holder", "")),
                "shares":       int(row.get("Shares", 0)) if pd.notna(row.get("Shares")) else 0,
                "date_reported":str(row.get("Date Reported", ""))[:10],
                "pct_out":      float(row.get("% Out", 0)) if pd.notna(row.get("% Out")) else 0,
                "value_usd":    float(row.get("Value", 0)) if pd.notna(row.get("Value")) else 0,
            })
        print(f"   ✅ Mutual Fund Holders: {len(results)} funds")
        return results
    except Exception as e:
        print(f"   ⚠️  Mutual fund holders: {e}")
        return []


# ── Scrape OpenInsider for Recent Bulk Trades ─────────────────────
def fetch_openinsider(ticker: str) -> list:
    """
    Scrape openinsider.com for recent insider cluster buys.
    This is the best free source for bulk insider activity.
    """
    try:
        url  = f"http://openinsider.com/screener?s={ticker}&o=&pl=&ph=&ll=&lh=&fd=180&fdr=&td=0&tdr=&fdlyl=&fdlyh=&daysago=&xp=1&xs=1&vl=&vh=&ocl=&och=&sic1=-1&sicl=100&sich=9999&grp=0&nfl=&nfh=&nil=&nih=&nol=&noh=&v2l=&v2h=&oc2l=&oc2h=&sortcol=0&cnt=40&action=1"
        resp = requests.get(url, headers=HEADERS, timeout=12)
        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table", class_="tinytable")
        if not table:
            return []

        rows = table.find_all("tr")[1:]
        results = []
        for row in rows[:20]:
            cols = row.find_all("td")
            if len(cols) < 12:
                continue
            results.append({
                "filing_date": cols[1].get_text(strip=True),
                "trade_date":  cols[2].get_text(strip=True),
                "insider":     cols[4].get_text(strip=True),
                "title":       cols[5].get_text(strip=True),
                "trade_type":  cols[6].get_text(strip=True),
                "price":       cols[7].get_text(strip=True),
                "qty":         cols[8].get_text(strip=True),
                "owned":       cols[9].get_text(strip=True),
                "delta_own":   cols[10].get_text(strip=True),
                "value":       cols[11].get_text(strip=True),
            })
        print(f"   ✅ OpenInsider: {len(results)} recent trades")
        return results
    except Exception as e:
        print(f"   ⚠️  OpenInsider scrape: {e}")
        return []


# ── Scrape SEC EDGAR Form 4 Filings ──────────────────────────────
def fetch_sec_form4(ticker: str) -> list:
    """
    Pull recent Form 4 filings from SEC EDGAR (official insider filings).
    Form 4 = mandatory insider transaction disclosures.
    """
    try:
        # Get CIK number for ticker
        search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{ticker}%22&dateRange=custom&startdt={(datetime.now()-timedelta(days=90)).strftime('%Y-%m-%d')}&enddt={TODAY}&forms=4"
        resp = requests.get(search_url, headers=HEADERS, timeout=10)
        data = resp.json()

        filings = []
        hits = data.get("hits", {}).get("hits", [])[:10]
        for hit in hits:
            src = hit.get("_source", {})
            filings.append({
                "form":        src.get("form_type", "4"),
                "filed":       src.get("file_date", ""),
                "entity":      src.get("entity_name", ""),
                "description": src.get("period_of_report", ""),
            })
        print(f"   ✅ SEC Form 4 filings: {len(filings)} recent")
        return filings
    except Exception as e:
        print(f"   ⚠️  SEC EDGAR: {e}")
        return []


# ── Calculate Insider Summary Stats ──────────────────────────────
def calculate_insider_stats(transactions: list, open_insider: list) -> dict:
    """Compute buy/sell ratios and net sentiment from raw data."""
    if not transactions and not open_insider:
        return {}

    buys   = [t for t in transactions if "buy"  in t.get("transaction","").lower() or "purchase" in t.get("transaction","").lower()]
    sells  = [t for t in transactions if "sell" in t.get("transaction","").lower() or "sale"     in t.get("transaction","").lower()]

    buy_value  = sum(t.get("value_usd", 0) for t in buys)
    sell_value = sum(t.get("value_usd", 0) for t in sells)
    total_val  = buy_value + sell_value

    # OpenInsider cluster buy signal
    oi_buys  = [o for o in open_insider if "P" in o.get("trade_type","") or "A" in o.get("trade_type","")]
    oi_sells = [o for o in open_insider if "S" in o.get("trade_type","")]

    return {
        "total_insider_trades":  len(transactions),
        "buy_count":             len(buys),
        "sell_count":            len(sells),
        "buy_value_usd":         round(buy_value, 2),
        "sell_value_usd":        round(sell_value, 2),
        "buy_sell_ratio":        round(buy_value / sell_value, 3) if sell_value > 0 else 99.0,
        "net_insider_flow_usd":  round(buy_value - sell_value, 2),
        "insider_sentiment":     "BULLISH" if buy_value > sell_value else "BEARISH" if sell_value > buy_value else "NEUTRAL",
        "cluster_buy_signal":    len(oi_buys) >= 3,  # 3+ insiders buying = strong signal
        "openinsider_buys":      len(oi_buys),
        "openinsider_sells":     len(oi_sells),
        "pct_buy_volume":        round(buy_value / total_val * 100, 1) if total_val > 0 else 0,
    }


# ════════════════════════════════════════════════════════════════════
# AGENT 4 — INSIDER & INSTITUTIONAL AI ANALYST
# ════════════════════════════════════════════════════════════════════

def agent4_insider_institutional(ticker: str, company: str,
                                  transactions: list, institutions: list,
                                  mutual_funds: list, open_insider: list,
                                  sec_filings: list, stats: dict) -> dict:
    """
    LLaMA 4 analyzes all insider + institutional data and produces
    a structured signal with confidence score.
    """
    print("\n🏦 Agent 4 — Insider & Institutional Analyst thinking...")

    # Format data for LLM
    txn_summary = "\n".join([
        f"  [{t['date']}] {t['insider']} ({t['title']}) → {t['transaction']} "
        f"{t['shares']:,} shares @ ~${t['value_usd']:,.0f} total"
        for t in transactions[:12]
    ]) or "No recent insider transactions found."

    inst_summary = "\n".join([
        f"  {i['holder']} → {i['shares']:,} shares ({i['pct_out']:.2f}% outstanding) "
        f"reported {i['date_reported']}"
        for i in institutions[:8]
    ]) or "No institutional data found."

    mf_summary = "\n".join([
        f"  {m['holder']} → {m['shares']:,} shares ({m['pct_out']:.2f}%)"
        for m in mutual_funds[:6]
    ]) or "No mutual fund data found."

    oi_summary = "\n".join([
        f"  [{o['trade_date']}] {o['insider']} ({o['title']}) → {o['trade_type']} "
        f"{o['qty']} shares worth {o['value']}"
        for o in open_insider[:10]
    ]) or "No OpenInsider data found."

    prompt = f"""
You are an elite hedge fund analyst specializing in insider trading signals
and institutional flow analysis. Your job is to interpret smart money movements.

STOCK: {ticker} ({company})
TODAY: {TODAY}
PREDICTION DATE: {DAY_AFTER_TMR}

═══ INSIDER TRANSACTION SUMMARY ═══
{txn_summary}

═══ INSIDER STATISTICS ═══
Total Trades: {stats.get('total_insider_trades', 0)}
Buy Count: {stats.get('buy_count', 0)} | Sell Count: {stats.get('sell_count', 0)}
Buy Value: ${stats.get('buy_value_usd', 0):,.0f} | Sell Value: ${stats.get('sell_value_usd', 0):,.0f}
Buy/Sell Ratio: {stats.get('buy_sell_ratio', 0)}
Net Insider Flow: ${stats.get('net_insider_flow_usd', 0):,.0f}
Cluster Buy Signal: {'YES ⚠️' if stats.get('cluster_buy_signal') else 'No'}
OpenInsider Buys: {stats.get('openinsider_buys', 0)} | Sells: {stats.get('openinsider_sells', 0)}

═══ TOP INSTITUTIONAL HOLDERS ═══
{inst_summary}

═══ TOP MUTUAL FUND HOLDERS ═══
{mf_summary}

═══ RECENT OPENINSIDER ACTIVITY ═══
{oi_summary}

═══ SEC FORM 4 FILINGS (Last 90 days) ═══
Count: {len(sec_filings)} recent filings

Analyze this smart money data carefully:
- Insiders know the company better than anyone
- Cluster buying (3+ insiders buying) is the strongest signal
- Large institutional accumulation = bullish confirmation
- CEO/CFO buying their own stock = very strong bullish signal
- Routine selling (10b5-1 plans) is less meaningful than open-market buys
- High buy/sell ratio > 1.5 is bullish; < 0.5 is bearish

Reply ONLY with valid JSON:
{{
  "insider_signal": "STRONG BULLISH" | "BULLISH" | "NEUTRAL" | "BEARISH" | "STRONG BEARISH",
  "insider_score": <float -1.0 to +1.0>,
  "confidence": <float 0.0 to 1.0>,
  "cluster_buy_detected": true | false,
  "smart_money_flow": "ACCUMULATING" | "DISTRIBUTING" | "NEUTRAL",
  "institutional_stance": "BULLISH" | "NEUTRAL" | "BEARISH",
  "key_insider_activity": "<most significant insider trade and who did it>",
  "notable_institutions": "<top 2-3 institutions and what they signal>",
  "red_flags": "<any concerning selling patterns or red flags, or None>",
  "bullish_signals": "<strongest bullish insider/institutional signals>",
  "insider_reasoning": "<2-3 sentences explaining what smart money is signaling>",
  "predicted_bias_{DAY_AFTER_TMR}": "UP" | "DOWN" | "SIDEWAYS"
}}
"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an elite insider trading and institutional flow analyst. Reply with valid JSON only."},
            {"role": "user",   "content": prompt}
        ],
        max_tokens=1024,
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
    try:
        result = json.loads(raw)
        print(f"   ✅ Insider Signal: {result.get('insider_signal')} | Score: {result.get('insider_score')} | Smart Money: {result.get('smart_money_flow')}")
        return result
    except:
        print(f"   ⚠️  JSON parse issue: {raw[:100]}")
        return {"insider_signal": "NEUTRAL", "insider_score": 0.0, "confidence": 0.3}


# ════════════════════════════════════════════════════════════════════
# UPDATED AGENT 3 — NOW COMBINES ALL 4 AGENTS
# ════════════════════════════════════════════════════════════════════

def agent3_master_synthesizer(ticker, company, sentiment, technical, insider, price_data):
    """
    Combines:
      Agent 1 (News)       → 30% weight
      Agent 2 (Technical)  → 40% weight
      Agent 4 (Insider)    → 30% weight
    """
    print("\n🧠 Agent 3 — Master Synthesizer (4-Agent Fusion)...")

    s_score = sentiment.get("sentiment_score", 0) * sentiment.get("confidence", 0.5)
    t_score = technical.get("technical_score", 0) * technical.get("confidence", 0.5)
    i_score = insider.get("insider_score",    0) * insider.get("confidence",   0.5)

    # Weights: 30% news, 40% technical, 30% insider
    combined = round(s_score * 0.30 + t_score * 0.40 + i_score * 0.30, 4)

    # Cluster buy override — strong signal
    cluster_boost = ""
    if insider.get("cluster_buy_detected"):
        combined = round(min(combined + 0.15, 1.0), 4)
        cluster_boost = "⚠️ CLUSTER BUY BOOST APPLIED (+0.15)"

    prompt = f"""
You are the Chief Investment Strategist synthesizing reports from 3 specialist AI agents.
Produce the FINAL definitive prediction for {ticker} on {DAY_AFTER_TMR}.

═══ AGENT 1 — NEWS SENTIMENT (30% weight) ═══
{json.dumps(sentiment, indent=2)}

═══ AGENT 2 — TECHNICAL ANALYSIS (40% weight) ═══
{json.dumps(technical, indent=2)}

═══ AGENT 4 — INSIDER & INSTITUTIONAL (30% weight) ═══
{json.dumps(insider, indent=2)}

═══ COMBINED WEIGHTED SCORE ═══
Formula: (News × 0.30) + (Technical × 0.40) + (Insider × 0.30) = {combined}
{cluster_boost}
Score interpretation: -1.0 = Strong Sell | 0 = Neutral | +1.0 = Strong Buy

CURRENT STOCK INFO:
Ticker: {ticker} | Company: {company}
Current Price: ${price_data.get('current_price')}
Sector: {price_data.get('sector')} | Beta: {price_data.get('beta')}

IMPORTANT RULES:
- If insiders are cluster buying AND technicals are bullish → upgrade verdict
- If insiders are selling heavily AND news is negative → downgrade verdict
- Conflicting signals = lower confidence, HOLD bias
- Smart money (insiders) often leads price by 2-4 weeks

Reply ONLY with valid JSON:
{{
  "ticker": "{ticker}",
  "company": "{company}",
  "prediction_date": "{DAY_AFTER_TMR}",
  "final_verdict": "STRONG BUY" | "BUY" | "HOLD" | "SELL" | "STRONG SELL",
  "final_direction": "UP" | "DOWN" | "SIDEWAYS",
  "combined_score": {combined},
  "overall_confidence": <float 0.0-1.0>,
  "price_now": {price_data.get('current_price')},
  "predicted_price": <float>,
  "predicted_change_%": <float>,
  "predicted_range_low": <float>,
  "predicted_range_high": <float>,
  "news_weight": 0.30,
  "technical_weight": 0.40,
  "insider_weight": 0.30,
  "news_summary": "<one sentence>",
  "technical_summary": "<one sentence>",
  "insider_summary": "<one sentence>",
  "all_agents_agree": true | false,
  "cluster_buy_boost": {"true" if insider.get("cluster_buy_detected") else "false"},
  "conflicting_signals": "<describe conflicts or None>",
  "primary_risk": "<biggest risk>",
  "primary_opportunity": "<biggest opportunity>",
  "final_reasoning": "<4-5 sentences synthesizing all 3 signals>",
  "disclaimer": "AI-generated analysis for educational purposes only. Not financial advice."
}}
"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are the world's best investment strategist. Reply with valid JSON only."},
            {"role": "user",   "content": prompt}
        ],
        max_tokens=1500,
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
    try:
        result = json.loads(raw)
        print(f"   ✅ Final Verdict: {result.get('final_verdict')} | Direction: {result.get('final_direction')} | Price Target: ${result.get('predicted_price')}")
        return result
    except:
        return {"final_verdict": "HOLD", "final_direction": "SIDEWAYS", "combined_score": combined}


# ════════════════════════════════════════════════════════════════════
# UPDATED DISPLAY REPORT (includes Agent 4)
# ════════════════════════════════════════════════════════════════════

def display_report(sentiment, technical, insider, verdict, price_data):
    v     = verdict.get("final_verdict", "HOLD")
    d     = verdict.get("final_direction", "SIDEWAYS")
    color = {"STRONG BUY":"#00C853","BUY":"#69F0AE","HOLD":"#FFD740",
             "SELL":"#FF6D00","STRONG SELL":"#D50000"}.get(v, "#FFD740")
    tc    = "#000" if v in ["HOLD","BUY","STRONG BUY","STRONG SELL"] else "#fff"
    emoji = {"UP":"📈","DOWN":"📉","SIDEWAYS":"➡️"}.get(d,"➡️")
    conf  = round(verdict.get("overall_confidence",0.5)*100,1)

    insider_color = {"STRONG BULLISH":"#00C853","BULLISH":"#69F0AE",
                     "NEUTRAL":"#FFD740","BEARISH":"#FF6D00",
                     "STRONG BEARISH":"#D50000"}.get(insider.get("insider_signal","NEUTRAL"),"#FFD740")

    cluster_badge = """<span style="background:#FF6D00;color:white;padding:3px 10px;
                       border-radius:12px;font-size:12px;font-weight:bold;">
                       ⚡ CLUSTER BUY DETECTED</span>""" if insider.get("cluster_buy_detected") else ""

    display(HTML(f"""
    <div style="font-family:Arial;max-width:900px;margin:auto;padding:10px;">

      <!-- Header -->
      <div style="background:#0D1B2A;color:white;padding:20px;border-radius:14px 14px 0 0;text-align:center;border-bottom:3px solid #00C853;">
        <h2 style="margin:0;font-size:22px;">🏦 4-AGENT STOCK ANALYSIS SYSTEM</h2>
        <p style="color:#aaa;margin:4px 0;">Generated: {TODAY} | Prediction: <b style="color:white">{DAY_AFTER_TMR}</b></p>
        <h3 style="color:#00d4ff;margin:6px 0;">{verdict.get('ticker')} — {verdict.get('company')}</h3>
        <p style="color:#aaa;">Current Price: <b style="color:white">${price_data.get('current_price')}</b>
           &nbsp;|&nbsp; Sector: {price_data.get('sector','N/A')}
           &nbsp;|&nbsp; Beta: {price_data.get('beta','N/A')}</p>
      </div>

      <!-- Verdict -->
      <div style="background:{color};color:{tc};padding:20px;text-align:center;">
        <h1 style="margin:0;font-size:36px;">{emoji} {v}</h1>
        <p style="font-size:18px;margin:8px 0;">
          ${price_data.get('current_price')} → <b>${verdict.get('predicted_price')}</b>
          (<b>{verdict.get('predicted_change_%')}%</b>)
        </p>
        <p>Range: ${verdict.get('predicted_range_low')} – ${verdict.get('predicted_range_high')}
           &nbsp;|&nbsp; Confidence: <b>{conf}%</b>
           &nbsp;|&nbsp; Score: <b>{verdict.get('combined_score')}</b></p>
        {cluster_badge}
      </div>

      <!-- Weight Bar -->
      <div style="background:#1B2A3A;padding:12px 16px;display:flex;gap:0;border-bottom:1px solid #2a3a4a;">
        <div style="flex:30;background:#2196F3;padding:8px;text-align:center;font-size:12px;border-radius:6px 0 0 6px;">
          📰 News 30%
        </div>
        <div style="flex:40;background:#9C27B0;padding:8px;text-align:center;font-size:12px;">
          📊 Technical 40%
        </div>
        <div style="flex:30;background:#FF6D00;padding:8px;text-align:center;font-size:12px;border-radius:0 6px 6px 0;">
          🏦 Insider 30%
        </div>
      </div>

      <!-- 3 Agent Cards -->
      <div style="display:flex;gap:10px;padding:12px;background:#f0f4f8;flex-wrap:wrap;">

        <!-- Agent 1 -->
        <div style="flex:1;min-width:220px;background:white;border-radius:10px;padding:14px;border-top:4px solid #2196F3;">
          <h4 style="color:#2196F3;margin:0 0 8px;">📰 Agent 1 — News</h4>
          <p><b>Sentiment:</b> {sentiment.get('overall_sentiment')}</p>
          <p><b>Score:</b> {sentiment.get('sentiment_score')} | <b>Conf:</b> {round(sentiment.get('confidence',0)*100)}%</p>
          <p><b>Momentum:</b> {sentiment.get('news_momentum','—')}</p>
          <p style="font-size:11px;color:#555;margin-top:6px;">{sentiment.get('reasoning','')[:120]}...</p>
        </div>

        <!-- Agent 2 -->
        <div style="flex:1;min-width:220px;background:white;border-radius:10px;padding:14px;border-top:4px solid #9C27B0;">
          <h4 style="color:#9C27B0;margin:0 0 8px;">📊 Agent 2 — Technical</h4>
          <p><b>Bias:</b> {technical.get('technical_bias')}</p>
          <p><b>Trend:</b> {technical.get('trend','—')} | <b>RSI:</b> {technical.get('rsi_signal','—')}</p>
          <p><b>Risk:</b> {technical.get('risk','—')}</p>
          <p style="font-size:11px;color:#555;margin-top:6px;">{technical.get('reasoning','')[:120]}...</p>
        </div>

        <!-- Agent 4 -->
        <div style="flex:1;min-width:220px;background:white;border-radius:10px;padding:14px;border-top:4px solid #FF6D00;">
          <h4 style="color:#FF6D00;margin:0 0 8px;">🏦 Agent 4 — Insider</h4>
          <p><b>Signal:</b> <span style="color:{insider_color};font-weight:bold">{insider.get('insider_signal','—')}</span></p>
          <p><b>Smart Money:</b> {insider.get('smart_money_flow','—')}</p>
          <p><b>Cluster Buy:</b> {'⚡ YES' if insider.get('cluster_buy_detected') else '❌ No'}</p>
          <p style="font-size:11px;color:#555;margin-top:6px;">{insider.get('insider_reasoning','')[:120]}...</p>
        </div>

      </div>

      <!-- Insider Detail Panel -->
      <div style="background:white;margin:0 12px;padding:16px;border-radius:10px;margin-bottom:12px;border-left:4px solid #FF6D00;">
        <h4 style="color:#FF6D00;margin:0 0 10px;">🏦 Insider & Institutional Deep Dive</h4>
        <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;">
          <div>
            <p><b>Key Insider Activity:</b></p>
            <p style="font-size:13px;color:#333;">{insider.get('key_insider_activity','—')}</p>
            <p style="margin-top:8px;"><b>Notable Institutions:</b></p>
            <p style="font-size:13px;color:#333;">{insider.get('notable_institutions','—')}</p>
          </div>
          <div>
            <p><b>Bullish Signals:</b></p>
            <p style="font-size:13px;color:#2e7d32;">{insider.get('bullish_signals','—')}</p>
            <p style="margin-top:8px;"><b>Red Flags:</b></p>
            <p style="font-size:13px;color:#c62828;">{insider.get('red_flags','None')}</p>
          </div>
        </div>
      </div>

      <!-- Final Synthesis -->
      <div style="background:white;margin:0 12px 12px;padding:16px;border-radius:10px;">
        <h4>🧠 Agent 3 — Final 4-Signal Synthesis</h4>
        <p><b>Combined Score:</b> {verdict.get('combined_score')} (News 30% + Technical 40% + Insider 30%)</p>
        <p><b>All Agents Agree:</b> {'✅ Yes' if verdict.get('all_agents_agree') else '⚠️ Partial conflict'}</p>
        <p><b>Cluster Buy Boost:</b> {'⚡ Applied' if verdict.get('cluster_buy_boost') else 'Not triggered'}</p>
        <p><b>Risk:</b> {verdict.get('primary_risk','—')}</p>
        <p><b>Opportunity:</b> {verdict.get('primary_opportunity','—')}</p>
        <div style="background:#f9f9f9;padding:12px;border-radius:6px;border-left:3px solid #333;margin-top:10px;font-size:14px;line-height:1.6;">
          {verdict.get('final_reasoning','—')}
        </div>
      </div>

      <!-- Disclaimer -->
      <div style="background:#ffebee;padding:10px;border-radius:0 0 12px 12px;font-size:11px;color:#c62828;text-align:center;">
        ⚠️ {verdict.get('disclaimer','AI analysis for educational purposes only. Not financial advice.')}
      </div>

    </div>
    """))


# ════════════════════════════════════════════════════════════════════
# UPDATED MAIN ORCHESTRATOR — ALL 4 AGENTS
# ════════════════════════════════════════════════════════════════════

def analyze_stock(ticker):
    ticker = ticker.upper().strip()
    print(f"\n{'═'*60}")
    print(f"  🏦 4-AGENT STOCK ANALYSIS: {ticker}")
    print(f"  📅 Predicting for: {DAY_AFTER_TMR}")
    print(f"{'═'*60}")

    # ── Fetch price data ──────────────────────────────────────────
    print("\n⬇️  [1/6] Fetching price & fundamental data...")
    price_data = fetch_price_data(ticker)
    if not price_data:
        print(f"❌ Could not fetch data for {ticker}")
        return None
    company = price_data.get("company", ticker)
    print(f"   ✅ {company} | ${price_data['current_price']} | {price_data['sector']}")

    # ── Fetch news ────────────────────────────────────────────────
    print("\n⬇️  [2/6] Fetching latest news...")
    articles = fetch_news(ticker, company)

    # ── Fetch insider & institutional data ────────────────────────
    print("\n⬇️  [3/6] Fetching insider & institutional data...")
    transactions  = fetch_insider_transactions(ticker)
    institutions  = fetch_institutional_holders(ticker)
    mutual_funds  = fetch_mutualfund_holders(ticker)
    open_insider  = fetch_openinsider(ticker)
    sec_filings   = fetch_sec_form4(ticker)
    insider_stats = calculate_insider_stats(transactions, open_insider)

    # ── Agent 1: News Sentiment ───────────────────────────────────
    print("\n🤖 [4/6] Running Agent 1 — News Sentiment...")
    s = agent1(ticker, company, articles)
    time.sleep(1.5)

    # ── Agent 2: Technical ────────────────────────────────────────
    print("\n🤖 [5/6] Running Agent 2 — Technical Analysis...")
    t = agent2(ticker, price_data)
    time.sleep(1.5)

    # ── Agent 4: Insider & Institutional ─────────────────────────
    print("\n🤖 [5/6] Running Agent 4 — Insider & Institutional...")
    ins = agent4_insider_institutional(
        ticker, company, transactions, institutions,
        mutual_funds, open_insider, sec_filings, insider_stats
    )
    time.sleep(1.5)

    # ── Agent 3: Master Synthesizer ───────────────────────────────
    print("\n🤖 [6/6] Running Agent 3 — Master Synthesizer...")
    v = agent3_master_synthesizer(ticker, company, s, t, ins, price_data)

    # ── Display ───────────────────────────────────────────────────
    print(f"\n{'═'*60}\n  📋 FINAL 4-AGENT REPORT\n{'═'*60}\n")
    display_report(s, t, ins, v, price_data)

    return {
        "sentiment":    s,
        "technical":    t,
        "insider":      ins,
        "verdict":      v,
        "price_data":   price_data,
        "insider_stats": insider_stats,
    }


#print("✅ 4-Agent Stock Analyzer loaded!")
#print("   Agents: News · Technical · Insider/Institutional · Synthesizer")
#print("   Run: analyze_stock('AAPL')")

In [9]:
GROQ_KEY = ""   #paste your groq key here. # from console.groq.com
client = groq.Groq(api_key=GROQ_KEY)

In [11]:
analyze_stock("NKE")   # try any ticker


═══════════════════════════════════════════════════════
  🏦 ANALYZING: NKE | Target: 2026-05-15
═══════════════════════════════════════════════════════

⬇️  Fetching price data...
   ✅ NIKE, Inc. | $42.34

⬇️  Fetching news...
   📰 15 articles fetched

📰 Agent 1 — News Sentiment Analyst...


APIConnectionError: Connection error.